# Voice to Text (Whisper)

Record from mic until silence, transcribe with OpenAI Whisper. No ffmpeg needed: audio goes to Whisper as a numpy array.

**Fixes vs old notebook**
- ffmpeg dependency removed (numpy array path)
- `audioop` (removed in Python 3.13) replaced with numpy RMS
- Hindi no longer translated to English: `task="transcribe"` + explicit `LANGUAGE`
- Hallucination loops cut: `temperature=0`, `condition_on_previous_text=False`
- Silence threshold auto-calibrated from ambient noise (was hardcoded 500)
- Hard cap on recording length, mic released in `finally`
- Model loaded before recording so download never stalls the mic
- Any-rate WAV auto-resampled to 16 kHz
- Structured result: `text`, `language`, `segments` with timestamps, timings

In [1]:
# Run once, then restart kernel.
# %pip install openai-whisper pyaudio numpy scipy torch

In [2]:
%pip install openai-whisper pyaudio numpy scipy torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import time
import wave

import numpy as np
import pyaudio
import whisper

## Settings

In [4]:
SAMPLE_RATE = 16000            # Whisper native rate
CHANNELS = 1
CHUNK = 1024                   # ~64 ms per chunk at 16 kHz
SAMPLE_WIDTH = 2               # int16

CALIBRATION_SEC = 0.7          # sample ambient noise before recording
THRESHOLD_MULTIPLIER = 2.5     # voice = rms > noise_floor * this
MIN_THRESHOLD = 300            # never go below this (int16 rms)
SILENCE_LIMIT_SEC = 5.0        # stop after this much trailing silence
START_TIMEOUT_SEC = 10.0       # abort if no voice within this window
MAX_RECORD_SEC = 120.0         # hard cap

OUTPUT_FILE = "audio.wav"
MODEL_NAME = "base"            # tiny | base | small | medium | large
LANGUAGE = None                # "hi", "en", or None = auto-detect
# Vocabulary hint: names, college, tech terms. Whisper biases toward these spellings.
# In the interview system, build this from the resume + JD.
VOCAB_HINT = "Chaitanya Rana, Vishwakarma College of Engineering, SQL, inner join, left join, pandas, Python"
FP16 = False                   # must be False on CPU

_MODEL_CACHE: dict = {}

## Audio helpers

In [5]:
def rms(chunk: bytes) -> float:
    samples = np.frombuffer(chunk, dtype=np.int16).astype(np.float32)
    if samples.size == 0:
        return 0.0
    return float(np.sqrt(np.mean(samples ** 2)))


def pcm_to_float32(pcm: bytes) -> np.ndarray:
    """int16 PCM -> float32 in [-1, 1]. Whisper accepts this directly."""
    return np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768.0


def save_wav(pcm: bytes, path: str = OUTPUT_FILE) -> str:
    with wave.open(path, "wb") as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(SAMPLE_WIDTH)
        wf.setframerate(SAMPLE_RATE)
        wf.writeframes(pcm)
    return path


def load_wav_as_float32(path: str) -> np.ndarray:
    """Read any WAV -> float32 mono 16 kHz."""
    from scipy.io import wavfile

    rate, data = wavfile.read(path)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if data.dtype == np.int16:
        data = data.astype(np.float32) / 32768.0
    elif data.dtype == np.int32:
        data = data.astype(np.float32) / 2147483648.0
    else:
        data = data.astype(np.float32)
    if rate != SAMPLE_RATE:
        from scipy.signal import resample_poly

        data = resample_poly(data, SAMPLE_RATE, rate).astype(np.float32)
    return data

## Recording (silence auto-stop)

In [6]:
def _calibrate_threshold(stream) -> float:
    """Measure ambient noise for CALIBRATION_SEC, return voice threshold."""
    n_chunks = max(1, int(SAMPLE_RATE / CHUNK * CALIBRATION_SEC))
    levels = [rms(stream.read(CHUNK, exception_on_overflow=False)) for _ in range(n_chunks)]
    noise_floor = float(np.median(levels))
    return max(MIN_THRESHOLD, noise_floor * THRESHOLD_MULTIPLIER)


def record_until_silence(
    silence_limit: float = SILENCE_LIMIT_SEC,
    start_timeout: float = START_TIMEOUT_SEC,
    max_seconds: float = MAX_RECORD_SEC,
) -> bytes:
    """Record from default mic. Stops after `silence_limit` s of silence post-speech.

    Returns raw int16 PCM bytes; empty bytes if no voice detected in time.
    """
    audio = pyaudio.PyAudio()
    stream = audio.open(
        format=pyaudio.paInt16,
        channels=CHANNELS,
        rate=SAMPLE_RATE,
        input=True,
        frames_per_buffer=CHUNK,
    )

    frames: list[bytes] = []
    try:
        print("Calibrating mic (stay quiet)...", end=" ", flush=True)
        threshold = _calibrate_threshold(stream)
        print(f"threshold={threshold:.0f}")
        print(f"Recording. Speak now. Stops after {silence_limit:.0f}s silence.")

        voice_started = False
        silence_start = None
        start_time = time.time()

        while True:
            data = stream.read(CHUNK, exception_on_overflow=False)
            frames.append(data)
            now = time.time()

            if now - start_time > max_seconds:
                print("\nStopped (max length).")
                break

            if rms(data) > threshold:
                voice_started = True
                silence_start = None
                print(".", end="", flush=True)
                continue

            if voice_started:
                silence_start = silence_start or now
                silent_for = now - silence_start
                print(f"\rsilence {silent_for:.1f}s ", end="", flush=True)
                if silent_for >= silence_limit:
                    print("\nStopped (silence).")
                    break
            elif now - start_time > start_timeout:
                print("\nNo voice detected.")
                return b""
    finally:
        stream.stop_stream()
        stream.close()
        audio.terminate()

    return b"".join(frames)

## Transcription

In [7]:
def load_model(name: str = MODEL_NAME):
    if name not in _MODEL_CACHE:
        print(f"Loading Whisper '{name}'...", flush=True)
        _MODEL_CACHE[name] = whisper.load_model(name)
    return _MODEL_CACHE[name]


def transcribe(
    audio: np.ndarray,
    language: str | None = LANGUAGE,
    model_name: str = MODEL_NAME,
    vocab_hint: str = VOCAB_HINT,
) -> dict:
    """audio: float32 mono 16 kHz in [-1, 1].

    Returns:
        {
          "text": str,
          "language": str,
          "duration_sec": float,
          "transcribe_sec": float,
          "segments": [{"start": float, "end": float, "text": str}, ...],
        }
    """
    model = load_model(model_name)
    t0 = time.time()
    result = model.transcribe(
        audio,
        language=language,
        task="transcribe",                 # never "translate": keep source language
        fp16=FP16,
        temperature=0.0,                   # deterministic, less hallucination
        condition_on_previous_text=False,  # stops repeat-loop hallucinations
        initial_prompt=vocab_hint or None,     # bias spelling of names / domain terms
    )
    return {
        "text": result["text"].strip(),
        "language": result.get("language"),
        "duration_sec": round(len(audio) / SAMPLE_RATE, 2),
        "transcribe_sec": round(time.time() - t0, 2),
        "segments": [
            {"start": round(s["start"], 2), "end": round(s["end"], 2), "text": s["text"].strip()}
            for s in result.get("segments", [])
        ],
    }


def record_and_transcribe(
    language: str | None = LANGUAGE,
    model_name: str = MODEL_NAME,
    save_path: str | None = OUTPUT_FILE,
    vocab_hint: str = VOCAB_HINT,
) -> dict | None:
    """One-shot: record until silence, then transcribe. Importable entrypoint."""
    load_model(model_name)  # warm up first so model download never stalls the mic
    pcm = record_until_silence()
    if not pcm:
        return None
    if save_path:
        print(f"Saved {save_wav(pcm, save_path)}")
    return transcribe(pcm_to_float32(pcm), language=language, model_name=model_name, vocab_hint=vocab_hint)


def show(result):
    if result is None:
        print("(no audio)")
        return
    print("\n=== TRANSCRIPT ===")
    print(result["text"] or "(empty)")
    print(f"=== lang={result['language']}  audio={result['duration_sec']}s  took={result['transcribe_sec']}s ===")

## Run: record from mic

In [13]:
# language="hi" forces Hindi (no auto-detect). model_name="small" is better for Hinglish.
result = record_and_transcribe(language=LANGUAGE, model_name=MODEL_NAME)
show(result)

Calibrating mic (stay quiet)... threshold=300
Recording. Speak now. Stops after 5s silence.
silence 5.1s ..................................................................................................
Stopped (silence).
Saved audio.wav

=== TRANSCRIPT ===
Hello, my name is Sethanikurana and I am a student of the William of College of Engineering and the answer to this question is we will use Inner Join, Aki, it is not a problem to have a enjoying car ride in the road.
=== lang=en  audio=26.43s  took=3.96s ===


## Run: transcribe an existing WAV file

In [9]:
# audio = load_wav_as_float32("audio.wav")
# show(transcribe(audio, language=LANGUAGE))

## Structured output (segments + timestamps for downstream)

In [10]:
# print(json.dumps(result, ensure_ascii=False, indent=2))

## Real-time (streaming) transcription

Mic runs continuously. Each spoken phrase is cut on a ~0.8 s pause and sent to a background Whisper worker; text prints as soon as each phrase is done, while you keep talking. Stops after 5 s of silence.

- `PAUSE_SEC` — gap that ends a phrase (lower = faster feedback, choppier text)
- `MAX_SEGMENT_SEC` — force-cut long monologues
- `initial_prompt` carries previous text into the next segment for vocabulary continuity
- Segments Whisper marks as `no_speech_prob > 0.6` are dropped (hallucination guard)

In [15]:
import queue
import threading

PAUSE_SEC = 0.8            # gap that ends one spoken segment
MAX_SEGMENT_SEC = 15.0     # force-cut long monologues so text keeps flowing
MIN_SEGMENT_SEC = 0.4      # ignore clicks / breaths
NO_SPEECH_PROB = 0.6       # drop Whisper segments it thinks are silence


def _transcribe_segment(model, audio, language, prev_text):
    result = model.transcribe(
        audio,
        language=language,
        task="transcribe",
        fp16=FP16,
        temperature=0.0,
        condition_on_previous_text=False,
        initial_prompt=prev_text[-200:] or None,  # vocabulary continuity between segments
    )
    kept = [s for s in result.get("segments", []) if s.get("no_speech_prob", 0) < NO_SPEECH_PROB]
    return " ".join(s["text"].strip() for s in kept).strip(), result.get("language")


def realtime_transcribe(
    language=LANGUAGE,
    model_name=MODEL_NAME,
    pause_sec=PAUSE_SEC,
    end_silence_sec=SILENCE_LIMIT_SEC,
    start_timeout=START_TIMEOUT_SEC,
    max_seconds=MAX_RECORD_SEC,
    save_path=OUTPUT_FILE,
    vocab_hint=VOCAB_HINT,
):
    """Stream mic -> Whisper. Prints each segment as soon as it is transcribed.

    Returns dict: text, language, segments[{start, end, text}], duration_sec.
    """
    model = load_model(model_name)
    work = queue.Queue()
    done = threading.Event()
    pieces = []           # [{start, end, text}]
    detected = {"lang": None}

    def worker():
        prev = vocab_hint
        while True:
            item = work.get()
            if item is None:
                done.set()
                return
            seg_audio, t_start, t_end = item
            text, lang = _transcribe_segment(model, seg_audio, language, prev)
            if not text:
                continue
            detected["lang"] = detected["lang"] or lang
            pieces.append({"start": round(t_start, 2), "end": round(t_end, 2), "text": text})
            prev = (prev + " " + text)[-500:]
            print(f"[{t_start:5.1f}s] {text}", flush=True)

    threading.Thread(target=worker, daemon=True).start()

    audio = pyaudio.PyAudio()
    stream = audio.open(
        format=pyaudio.paInt16, channels=CHANNELS, rate=SAMPLE_RATE, input=True, frames_per_buffer=CHUNK
    )
    all_frames = []
    seg_frames = []
    chunk_sec = CHUNK / SAMPLE_RATE

    try:
        print("Calibrating mic (stay quiet)...", end=" ", flush=True)
        threshold = _calibrate_threshold(stream)
        print(f"threshold={threshold:.0f}")
        print(f"Live. Speak. Stops after {end_silence_sec:.0f}s silence.\n")

        elapsed = 0.0
        seg_start = 0.0
        voice_started = False
        in_voice = False
        silent_for = 0.0

        def flush_segment(end_t):
            nonlocal seg_frames, seg_start
            dur = end_t - seg_start
            if dur >= MIN_SEGMENT_SEC and seg_frames:
                work.put((pcm_to_float32(b"".join(seg_frames)), seg_start, end_t))
            seg_frames = []
            seg_start = end_t

        while elapsed < max_seconds:
            data = stream.read(CHUNK, exception_on_overflow=False)
            all_frames.append(data)
            elapsed += chunk_sec
            loud = rms(data) > threshold

            if loud:
                if not in_voice:
                    seg_start = elapsed - chunk_sec
                    seg_frames = []
                in_voice = True
                voice_started = True
                silent_for = 0.0
                seg_frames.append(data)
                if elapsed - seg_start >= MAX_SEGMENT_SEC:
                    flush_segment(elapsed)
                continue

            silent_for += chunk_sec
            if in_voice:
                seg_frames.append(data)  # keep short tail so words are not clipped
                if silent_for >= pause_sec:
                    in_voice = False
                    flush_segment(elapsed)
            if voice_started and silent_for >= end_silence_sec:
                print("\nStopped (silence).")
                break
            if not voice_started and elapsed >= start_timeout:
                print("\nNo voice detected.")
                break
        else:
            print("\nStopped (max length).")

        if in_voice:
            flush_segment(elapsed)
    finally:
        stream.stop_stream()
        stream.close()
        audio.terminate()

    work.put(None)
    done.wait()  # worker finishes queued segments, then signals

    pcm = b"".join(all_frames)
    if save_path and pcm:
        save_wav(pcm, save_path)

    pieces.sort(key=lambda p: p["start"])
    return {
        "text": " ".join(p["text"] for p in pieces),
        "language": detected["lang"],
        "duration_sec": round(len(pcm) / (SAMPLE_RATE * SAMPLE_WIDTH), 2),
        "segments": pieces,
    }

### Run: live

In [16]:
rt = realtime_transcribe(language=LANGUAGE, model_name=MODEL_NAME)
print("\n=== FULL ===")
print(rt["text"])
print(f"=== lang={rt['language']}  audio={rt['duration_sec']}s  segments={len(rt['segments'])} ===")

Calibrating mic (stay quiet)... threshold=300
Live. Speak. Stops after 5s silence.

[  1.5s] My name is Chetra Mithurana and I was...
[  4.5s] ...and I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the same way as I was born in the

Stopped (silence).
[ 